# 00_create_sample_dataset_for_embedding

A) Deduplicate intermonth from full Adzuna 2022 data.

B) Prepend Domains

C) Section B we remove core public sector jobs from Onet v30.0 reducing from 894 to 861 (drop 33 core public sector jobs). (THIS MUST BE CHECKED!!!)

In [ ]:
# lets transfer from previous generated and add domain to role_text in order to enrich it;

In [3]:
se for rodar tem rodar tudo de novo e apague todos os npz do stage2!
#!rm -f /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/**/*.npz


# Reducing dataset size deduplicating intermonth duplicates.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
start_time = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
output_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

all_data = []

for month in months:
    npz_file = src_root / f"{month}.npz"
    
    if not npz_file.exists():
        print(f"⚠️  Skipping {month} - file not found")
        continue
    
    # Load NPZ and extract job_ids
    with np.load(npz_file, allow_pickle=True) as data:
        job_ids = data["job_ids"].astype(str)
        
        # Create dataframe for this month
        df_month = pd.DataFrame({
            'id': job_ids,
            'npz_file': f"{month}.npz"
        })
        
        all_data.append(df_month)
        print(f"✓ {month}.npz: {len(job_ids):,} IDs")

# Concatenate all months
print(f"\nConcatenating {len(all_data)} files...")
df_combined = pd.concat(all_data, ignore_index=True)

print(f"Total rows before deduplication: {len(df_combined):,}")

# Sort by npz_file then drop duplicates by id
df_combined = df_combined.sort_values('npz_file').reset_index(drop=True)
rows_before = len(df_combined)
df_deduplicated = df_combined.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

duplicates_removed = rows_before - len(df_deduplicated)
print(f"Duplicates removed: {duplicates_removed:,}")
print(f"Final rows: {len(df_deduplicated):,}")

# Save
output_file.parent.mkdir(parents=True, exist_ok=True)
df_deduplicated.to_parquet(output_file, index=False)

duration = time.time() - start_time
print(f"\nSaved to: {output_file}")
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {duration:.2f}s ({duration/60:.2f}m)")

Started at: 2026-02-16 21:37:18

✓ adzuna_month01.npz: 2,657,586 IDs
✓ adzuna_month02.npz: 2,229,436 IDs
✓ adzuna_month03.npz: 2,303,805 IDs
✓ adzuna_month04.npz: 2,331,704 IDs
✓ adzuna_month05.npz: 2,546,484 IDs
✓ adzuna_month06.npz: 1,992,787 IDs
✓ adzuna_month07.npz: 2,183,287 IDs
✓ adzuna_month08.npz: 1,913,920 IDs
✓ adzuna_month09.npz: 1,860,287 IDs
✓ adzuna_month10.npz: 2,179,371 IDs
✓ adzuna_month11.npz: 2,040,587 IDs
✓ adzuna_month12.npz: 1,739,463 IDs
✓ adzuna_month13.npz: 2,537,947 IDs
✓ adzuna_month14.npz: 21,923 IDs

Concatenating 14 files...
Total rows before deduplication: 28,538,587
Duplicates removed: 8,857,090
Final rows: 19,681,497

Saved to: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet
Completed at: 2026-02-16 21:37:56
Duration: 38.06s (0.63m)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
start_time = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
parquet_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

# Load the deduplicated IDs
df_deduplicated = pd.read_parquet(parquet_file)
print(f"Loaded deduplicated parquet: {len(df_deduplicated):,} unique IDs\n")

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

total_original = 0
total_filtered = 0

for month in months:
    npz_file = src_root / f"{month}.npz"
    
    if not npz_file.exists():
        print(f"⚠️  Skipping {month} - file not found")
        continue
    
    # Load NPZ
    with np.load(npz_file, allow_pickle=True) as data:
        job_ids_original = data["job_ids"].astype(str)
        n_original = len(job_ids_original)
        
        # Get filtered IDs for this month
        filtered_ids = df_deduplicated[df_deduplicated['npz_file'] == f"{month}.npz"]['id'].values
        n_filtered = len(filtered_ids)
        
        # Calculate reduction
        reduction = n_original - n_filtered
        reduction_pct = (reduction / n_original * 100) if n_original > 0 else 0
        
        total_original += n_original
        total_filtered += n_filtered
        
        print(f"{month}.npz:")
        print(f"  Original: {n_original:,}")
        print(f"  Filtered: {n_filtered:,}")
        print(f"  Reduction: {reduction:,} ({reduction_pct:.2f}%)\n")

# Overall stats
total_reduction = total_original - total_filtered
total_reduction_pct = (total_reduction / total_original * 100) if total_original > 0 else 0

print("=" * 50)
print(f"TOTAL ACROSS ALL MONTHS:")
print(f"  Original: {total_original:,}")
print(f"  Filtered: {total_filtered:,}")
print(f"  Reduction: {total_reduction:,} ({total_reduction_pct:.2f}%)")

duration = time.time() - start_time
print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Duration: {duration:.2f}s ({duration/60:.2f}m)")

Started at: 2026-02-16 21:46:56

Loaded deduplicated parquet: 19,681,497 unique IDs

adzuna_month01.npz:
  Original: 2,657,586
  Filtered: 2,657,586
  Reduction: 0 (0.00%)

adzuna_month02.npz:
  Original: 2,229,436
  Filtered: 1,539,063
  Reduction: 690,373 (30.97%)

adzuna_month03.npz:
  Original: 2,303,805
  Filtered: 1,684,605
  Reduction: 619,200 (26.88%)

adzuna_month04.npz:
  Original: 2,331,704
  Filtered: 1,500,944
  Reduction: 830,760 (35.63%)

adzuna_month05.npz:
  Original: 2,546,484
  Filtered: 1,845,262
  Reduction: 701,222 (27.54%)

adzuna_month06.npz:
  Original: 1,992,787
  Filtered: 1,287,469
  Reduction: 705,318 (35.39%)

adzuna_month07.npz:
  Original: 2,183,287
  Filtered: 1,484,251
  Reduction: 699,036 (32.02%)

adzuna_month08.npz:
  Original: 1,913,920
  Filtered: 1,296,837
  Reduction: 617,083 (32.24%)

adzuna_month09.npz:
  Original: 1,860,287
  Filtered: 1,269,039
  Reduction: 591,248 (31.78%)

adzuna_month10.npz:
  Original: 2,179,371
  Filtered: 1,486,281
  R

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import time
from datetime import datetime
import gc

print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
t0 = time.time()

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")

out_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated_NO_DOMAIN")
out_root.mkdir(parents=True, exist_ok=True)

parquet_file = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet")

df = pd.read_parquet(parquet_file)
df["id"] = df["id"].astype(str)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

total_original = 0
total_filtered = 0

for month in months:
    npz_path = src_root / f"{month}.npz"
    out_path = out_root / f"{month}.npz"

    if not npz_path.exists():
        print(f"Skipping {month}: source file not found")
        continue

    keep_ids = df.loc[df["npz_file"] == f"{month}.npz", "id"].to_numpy(dtype=str)

    if keep_ids.size == 0:
        print(f"Skipping {month}: no IDs in dedup parquet")
        continue

    with np.load(npz_path, allow_pickle=True) as data:
        job_ids = data["job_ids"].astype(str)
        n_original = job_ids.shape[0]

        if keep_ids.size > n_original:
            # Not necessarily wrong, but a good smoke test
            print(f"Warning {month}: keep_ids ({keep_ids.size:,}) > job_ids ({n_original:,})")

        mask = np.isin(job_ids, keep_ids)
        n_filtered = int(mask.sum())

        if n_filtered == 0:
            raise RuntimeError(f"{month}: mask empty (no overlap between NPZ job_ids and dedup IDs)")

        filtered_data = {}
        for key in data.files:
            arr = data[key]
            if getattr(arr, "shape", None) is None or arr.shape[0] != n_original:
                raise ValueError(f"{month}: array '{key}' first-dim {getattr(arr,'shape',None)} != job_ids {n_original}")
            filtered_data[key] = arr[mask]

    np.savez_compressed(out_path, **filtered_data)

    total_original += n_original
    total_filtered += n_filtered

    reduction = n_original - n_filtered
    reduction_pct = reduction / n_original * 100

    del filtered_data, job_ids, mask, keep_ids
    gc.collect()

    print(f"{month}.npz")
    print(f"  Original : {n_original:,}")
    print(f"  Filtered : {n_filtered:,}")
    print(f"  Dropped  : {reduction:,} ({reduction_pct:.2f}%)")
    print(f"  Saved to : {out_path}\n")

dt = time.time() - t0
total_reduction = total_original - total_filtered
total_reduction_pct = total_reduction / total_original * 100 if total_original else 0

print("=" * 60)
print("TOTAL")
print(f"Original : {total_original:,}")
print(f"Filtered : {total_filtered:,}")
print(f"Dropped  : {total_reduction:,} ({total_reduction_pct:.2f}%)")
print(f"Duration : {dt:.2f}s ({dt/60:.2f}m)")
print(f"Finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Started at: 2026-02-17 22:20:48

adzuna_month01.npz
  Original : 2,657,586
  Filtered : 2,657,586
  Dropped  : 0 (0.00%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month01.npz

adzuna_month02.npz
  Original : 2,229,436
  Filtered : 1,539,063
  Dropped  : 690,373 (30.97%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month02.npz

adzuna_month03.npz
  Original : 2,303,805
  Filtered : 1,684,605
  Dropped  : 619,200 (26.88%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated/adzuna_month03.npz

adzuna_month04.npz
  Original : 2,331,704
  Filtered : 1,500,944
  Dropped  : 830,760 (35.63%)
  Saved to : /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store

Post-hoc NPZ verification script

What it checks:

file exists

job_ids present

all arrays have same first dimension

row count matches deduplicated parquet for that month

no silent truncation

In [6]:



import numpy as np
import pandas as pd
from pathlib import Path

src_root = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/"
    "stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated"
)

parquet_file = Path(   
    "/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/"
    "AISI_demo/stage_2_embeddings_and_cosines/npz_job_ids.parquet"
)

df = pd.read_parquet(parquet_file)
df["id"] = df["id"].astype(str)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

errors = 0

for month in months:
    npz_path = src_root / f"{month}.npz"

    if not npz_path.exists():
        print(f"ERROR {month}: output NPZ missing")
        errors += 1
        continue

    expected_n = df.loc[
        df["npz_file"] == f"{month}.npz", "id"
    ].nunique()

    with np.load(npz_path, allow_pickle=True) as data:
        if "job_ids" not in data.files:
            print(f"ERROR {month}: job_ids missing")
            errors += 1
            continue

        job_ids = data["job_ids"].astype(str)
        n = job_ids.shape[0]

        if n != expected_n:
            print(
                f"ERROR {month}: row count mismatch "
                f"(npz={n:,}, parquet={expected_n:,})"
            )
            errors += 1

        for key in data.files:
            arr = data[key]
            if arr.shape[0] != n:
                print(
                    f"ERROR {month}: array '{key}' "
                    f"has shape {arr.shape}, expected {n}"
                )
                errors += 1

    print(f"OK {month}: {n:,} rows")

print("=" * 60)
if errors == 0:
    print("ALL CHECKS PASSED")
else:
    print(f"FAILED WITH {errors} ERRORS")


OK adzuna_month01: 2,657,586 rows
OK adzuna_month02: 1,539,063 rows
OK adzuna_month03: 1,684,605 rows
OK adzuna_month04: 1,500,944 rows
OK adzuna_month05: 1,845,262 rows
OK adzuna_month06: 1,287,469 rows
OK adzuna_month07: 1,484,251 rows
OK adzuna_month08: 1,296,837 rows
OK adzuna_month09: 1,269,039 rows
OK adzuna_month10: 1,486,281 rows
OK adzuna_month11: 1,419,300 rows
OK adzuna_month12: 982,172 rows
OK adzuna_month13: 1,220,262 rows
OK adzuna_month14: 8,426 rows
ALL CHECKS PASSED


# Prepend Domain in role and tasks

In [4]:
import numpy as np
from pathlib import Path
import time

print("--- STARTING FULL DATASET RUN ---")

# === PATH CONFIGURATION ===
src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated_NO_DOMAIN")
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated")

dst_root.mkdir(parents=True, exist_ok=True)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

def prepend_domain(text_arr, domain_arr):
    # This creates a new list in memory. For 1M+ rows, this is RAM heavy.
    out = []
    # We iterate directly over the mmap arrays to avoid loading source twice
    for t, d in zip(text_arr, domain_arr):
        t = "" if t is None else str(t).strip()
        d = "" if d is None else str(d).strip()
        out.append(f"[{d}] {t}" if d else t)
    return np.array(out, dtype=object)

for month in months:
    start_time = time.perf_counter()
    
    src_file = src_root / f"{month}.npz"
    dst_file = dst_root / f"{month}.npz"

    if not src_file.exists():
        print(f"Skipping {month}: Not found")
        continue

    # Load with mmap (Crucial for memory management)
    with np.load(src_file, allow_pickle=True, mmap_mode="r") as data:
        keys = list(data.files)
        
        # 1. Load everything as a reference first (Lazy loading where possible)
        # Note: We use data[k] directly, not data[k][:], to keep it as a file reference until needed
        sampled = {k: data[k] for k in keys} 
        
        n_total = len(data[keys[0]])

        # 2. MODIFY TEXT (This forces the text columns into RAM)
        if "domains" in sampled:
            # We access the mmap arrays directly in the function arguments
            if "role_text" in sampled:
                print(f"  Prepending domains to role_text ({n_total} rows)...")
                sampled["role_text"] = prepend_domain(sampled["role_text"], sampled["domains"])
            
            if "taskskill_text" in sampled:
                print(f"  Prepending domains to taskskill_text ({n_total} rows)...")
                sampled["taskskill_text"] = prepend_domain(sampled["taskskill_text"], sampled["domains"])

        # 3. SAVE FULL DATASET
        print(f"  Saving to {dst_file.name}...")
        np.savez_compressed(dst_file, **sampled)

    end_time = time.perf_counter()
    print(f"Processed {month}: {n_total} rows (FULL)")
    print(f"  Time: {end_time - start_time:.4f}s")
    print("-" * 40)

--- STARTING FULL DATASET RUN ---
  Prepending domains to role_text (2657586 rows)...
  Prepending domains to taskskill_text (2657586 rows)...
  Saving to adzuna_month01.npz...
Processed adzuna_month01: 2657586 rows (FULL)
  Time: 51.9629s
----------------------------------------
  Prepending domains to role_text (1539063 rows)...
  Prepending domains to taskskill_text (1539063 rows)...
  Saving to adzuna_month02.npz...
Processed adzuna_month02: 1539063 rows (FULL)
  Time: 30.4236s
----------------------------------------
  Prepending domains to role_text (1684605 rows)...
  Prepending domains to taskskill_text (1684605 rows)...
  Saving to adzuna_month03.npz...
Processed adzuna_month03: 1684605 rows (FULL)
  Time: 32.9751s
----------------------------------------
  Prepending domains to role_text (1500944 rows)...
  Prepending domains to taskskill_text (1500944 rows)...
  Saving to adzuna_month04.npz...
Processed adzuna_month04: 1500944 rows (FULL)
  Time: 30.0833s
-------------------

In [ ]:
# VERIFY

In [7]:
import numpy as np
from pathlib import Path
import random

# Path configuration
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/llama_summary_deduplicated")
months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

def nonempty_share(arr):
    # Works on the already extracted array
    return sum(bool(str(v).strip()) for v in arr) / len(arr) if len(arr) else 0.0

def is_prefixed(text, domain):
    t = "" if text is None else str(text).strip()
    d = "" if domain is None else str(domain).strip()
    if not d:
        return True  # nothing to check
    return t.startswith(f"[{d}]")

print("Evaluating sampled outputs...\n")

for month in months:
    p = dst_root / f"{month}.npz"
    if not p.exists():
        print(f"{month}: MISSING -> {p}")
        continue

    # Load file
    # Note: For object arrays (strings), mmap_mode has limits, but we keep it to avoid eager loading if possible.
    try:
        d = np.load(p, allow_pickle=True, mmap_mode="r")
        keys = list(d.files)
    except Exception as e:
        print(f"{month}: FAILED TO LOAD -> {e}")
        continue

    need = ["job_ids", "role_text", "taskskill_text", "domains"]
    missing = [k for k in need if k not in keys]
    if missing:
        print(f"{month}: missing keys {missing} (has {keys})")
        continue

    n = len(d["job_ids"])
    
    # OPTIMIZATION 1: Sort indices to minimize disk seek time
    idxs = sorted(random.sample(range(n), min(200, n)))

    # OPTIMIZATION 2: Batch read. 
    # Instead of reading inside the loop 200 times, we read ONCE per column.
    # This forces the disk I/O to happen in one chunk.
    try:
        batch_roles = d["role_text"][idxs]
        batch_tasks = d["taskskill_text"][idxs]
        batch_domains = d["domains"][idxs]
    except Exception as e:
        print(f"{month}: Error reading batch -> {e}")
        continue

    # metric: non-empty share
    role_ne = nonempty_share(batch_roles)
    task_ne = nonempty_share(batch_tasks)
    dom_ne  = nonempty_share(batch_domains)

    # metric: prefix correctness
    # We zip the small batch arrays in memory (very fast)
    role_pref_ok = sum(is_prefixed(r, dom) for r, dom in zip(batch_roles, batch_domains)) / len(idxs)
    task_pref_ok = sum(is_prefixed(t, dom) for t, dom in zip(batch_tasks, batch_domains)) / len(idxs)

    # metric: lengths
    role_len = [len(str(r).strip()) for r in batch_roles]
    task_len = [len(str(t).strip()) for t in batch_tasks]

    print(f"{month}: n={n}")
    print(f"  non-empty share (sample200): role_text={role_ne:.3f} taskskill_text={task_ne:.3f} domains={dom_ne:.3f}")
    print(f"  prefix correctness (sample200): role_text={role_pref_ok:.3f} taskskill_text={task_pref_ok:.3f}")
    print(f"  length stats (sample200): role_text p50={int(np.median(role_len))} p95={int(np.quantile(role_len,0.95))} | taskskill_text p50={int(np.median(task_len))} p95={int(np.quantile(task_len,0.95))}")
    
    print(f"  example:")
    # We just grab the first item of our batch (which corresponds to idxs[0])
    print(f"    domain: {batch_domains[0]}")
    print(f"    role_text: {str(batch_roles[0])[:200]}")
    print(f"    taskskill_text: {str(batch_tasks[0])[:200]}")
    print()

Evaluating sampled outputs...

adzuna_month01: n=2657586
  non-empty share (sample200): role_text=1.000 taskskill_text=1.000 domains=1.000
  prefix correctness (sample200): role_text=1.000 taskskill_text=1.000
  length stats (sample200): role_text p50=84 p95=152 | taskskill_text p50=347 p95=486
  example:
    domain: Event Management
    role_text: [Event Management] Production & Content Administrator
    taskskill_text: [Event Management] Support the team in the management of all freelance contracting requirements and associated administration Researching new freelance support Contacting Freelance resource for availa

adzuna_month02: n=1539063
  non-empty share (sample200): role_text=1.000 taskskill_text=1.000 domains=1.000
  prefix correctness (sample200): role_text=1.000 taskskill_text=1.000
  length stats (sample200): role_text p50=87 p95=152 | taskskill_text p50=349 p95=508
  example:
    domain: IT and Technology
    role_text: [IT and Technology] IT Manager position for a growin

# C) Drop core state functions (not adversided in market such politicians judges etc).

Out of scope occupations in onet
For the transformer model in the onet v30.0 positions, I drop 40 non-private, core state roles (e.g. magistrates, police). These aren’t part of the market job space and were adding noise.

Statutory vs. Market Roles

Key Statutory Categories dropped: Judicial, Policing/Security, Emergency Services, Corrections, Postal Monopolies, and Sovereign Infrastructure.

Total exclusions 33 out of 894 onet v30.0 occupations

Final set is 861 occupations.

In [9]:
import shutil
from pathlib import Path
import os

# Define the Source (DEV) and Destination (PROD) directories
# I assumed the CSV source is 'dev' to match the NPZ pattern
src_dir_dev = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings")
dst_dir_prod = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings")

# List of files to copy (filename, source_folder, destination_folder)
files_to_copy = [
    {
        "name": "standard_df_onet_occupations_description_activities_and_tasks.csv",
        "src": src_dir_dev / "standard_df_onet_occupations_description_activities_and_tasks.csv",
        "dst": dst_dir_prod / "standard_df_onet_occupations_description_activities_and_tasks.csv"
    },
    {
        "name": "aspectt_vectors.npz",
        "src": Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz"),
        "dst": Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors.npz")
    }
]

print("Starting file copy...\n")

for item in files_to_copy:
    src = item["src"]
    dst = item["dst"]
    
    # 1. Check if source exists
    if not src.exists():
        print(f"❌ MISSING SOURCE: {src}")
        continue

    # 2. Ensure destination directory exists
    dst.parent.mkdir(parents=True, exist_ok=True)

    # 3. Copy the file
    try:
        shutil.copy2(src, dst)  # copy2 preserves metadata (timestamps)
        print(f"✅ Copied: {item['name']}")
        print(f"   From: {src}")
        print(f"   To:   {dst}\n")
    except Exception as e:
        print(f"❌ Error copying {item['name']}: {e}\n")

print("Done.")

Starting file copy...

✅ Copied: standard_df_onet_occupations_description_activities_and_tasks.csv
   From: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/standard_df_onet_occupations_description_activities_and_tasks.csv
   To:   /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/standard_df_onet_occupations_description_activities_and_tasks.csv

✅ Copied: aspectt_vectors.npz
   From: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz
   To:   /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors.npz

Done.


In [10]:
from pathlib import Path
import urllib.request

dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/")
dst = dst_root / "db_30_0_text.zip"

if not dst.exists():
    dst_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://www.onetcenter.org/dl_files/database/db_30_0_text.zip",
        dst
    )
from zipfile import ZipFile
import pandas as pd

with ZipFile(dst) as z:
    onet_soc_codes = set(
        pd.read_csv(
            z.open("db_30_0_text/Abilities.txt"),
            sep="\t",
            dtype=str
        )["O*NET-SOC Code"]
    )
standard_df_onet_occupations_description_activities_and_tasks = pd.read_csv( dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv")
check_1= set(standard_df_onet_occupations_description_activities_and_tasks['O*NET-SOC Code'])
print('same onet codes for onet 30.0?',len(onet_soc_codes.intersection(check_1)), ' IF RESPONSE is  894 with core public sector occs or 861 dropped core public sector jobs')

from pathlib import Path
import numpy as np

path = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors.npz")
data = np.load(path,allow_pickle=True)
print(data.files)
print('same onet titles for onet 30.0?',len(set(data['titles']).intersection(set(standard_df_onet_occupations_description_activities_and_tasks['Title']))), ' IF RESPONSE is 894 with core public sector occs or 861 dropped core public sector jobs')

same onet codes for onet 30.0? 861  IF RESPONSE is  894 with core public sector occs or 861 dropped core public sector jobs
['titles', 'columns', 'levels', 'importance']
same onet titles for onet 30.0? 861  IF RESPONSE is 894 with core public sector occs or 861 dropped core public sector jobs


In [ ]:
# if above is not 861 run the 33 positions drop:

In [11]:
import pandas as pd
list_core_public_state_related_jobs = {
  "definition": {
    "core_public_sector": "Roles with statutory authority, coercive power, public monopoly delivery, or security-vetted state functions. These are typically recruited via government portals/appointments rather than commercial job boards like Adzuna."
  },
  "core_not_expected_in_adzuna_from_your_list": {
    "justice_courts_statutory_legal": [
      "Judges, Magistrate Judges, and Magistrates",
      "Administrative Law Judges, Adjudicators, and Hearing Officers",
      "Judicial Law Clerks",
      "Coroners",
      "Bailiffs",
      "Probation Officers and Correctional Treatment Specialists",
      "Court, Municipal, and License Clerks",
      "Tax Examiners and Collectors, and Revenue Agents"
    ],
    "policing_intelligence_border_security": [
      "First-Line Supervisors of Police and Detectives",
      "Detectives and Criminal Investigators",
      "Police Identification and Records Officers",
      "Intelligence Analysts",
      "Fish and Game Wardens",
      "Parking Enforcement Workers",
      "Police and Sheriff's Patrol Officers",
      "Customs and Border Protection Officers",
      "Transit and Railroad Police",
      "Transportation Security Screeners"
    ],
    "fire_emergency_management_public_safety": [
      "Emergency Management Directors",
      "First-Line Supervisors of Firefighting and Prevention Workers",
      "Firefighters",
      "Fire Inspectors and Investigators",
      "Forest Fire Inspectors and Prevention Specialists",
      "Public Safety Telecommunicators"
    ],
    "corrections_detention": [
      "First-Line Supervisors of Correctional Officers",
      "Correctional Officers and Jailers"
    ],
    "postal_service_monopoly": [
      "Postmasters and Mail Superintendents",
      "Postal Service Clerks",
      "Postal Service Mail Carriers",
      "Postal Service Mail Sorters, Processors, and Processing Machine Operators"
    ],
    "government_programs_admin_gatekeeping": [
      "Eligibility Interviewers, Government Programs"
    ],
    "statutory_public_inspection_enforcement": [
      "Government Property Inspectors and Investigators",
    ],
    "sovereign_infrastructure_control": [
      "Air Traffic Controllers",
    ]
  },
  "notes": {
    "not_included_as_core": [
      "Teachers, nurses, clinicians, and university staff can be public sector but do commonly advertise via job boards depending on country and institution.",
      "Compliance, inspection, regulation, and security roles exist in private sector too. Only the explicitly state-linked statutory variants above were tagged core here.",
      "If you want a stricter version, we can add public-school teaching roles to core. If you want a looser version, we can remove court clerks and keep only judges/ALJs."
    ]
  }
}
import pandas as pd

# 1) core titles only (ignore definition + notes)
core_titles = [
    t.strip()
    for group in list_core_public_state_related_jobs["core_not_expected_in_adzuna_from_your_list"].values()
    for t in group
]

# 2) titles in your npz
series_set = set(pd.Series(data["titles"]).dropna().astype(str).str.strip())

# 3) compare
missing = [t for t in core_titles if t not in series_set]
if missing:
    raise ValueError(f"{len(missing)} missing. Examples: {missing[:20]}")
print(f"OK: {len(core_titles)} core titles present in data['titles'].")


ValueError: 33 missing. Examples: ['Judges, Magistrate Judges, and Magistrates', 'Administrative Law Judges, Adjudicators, and Hearing Officers', 'Judicial Law Clerks', 'Coroners', 'Bailiffs', 'Probation Officers and Correctional Treatment Specialists', 'Court, Municipal, and License Clerks', 'Tax Examiners and Collectors, and Revenue Agents', 'First-Line Supervisors of Police and Detectives', 'Detectives and Criminal Investigators', 'Police Identification and Records Officers', 'Intelligence Analysts', 'Fish and Game Wardens', 'Parking Enforcement Workers', "Police and Sheriff's Patrol Officers", 'Customs and Border Protection Officers', 'Transit and Railroad Police', 'Transportation Security Screeners', 'Emergency Management Directors', 'First-Line Supervisors of Firefighting and Prevention Workers']

In [13]:
# CSV
old_csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
new_csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks_full_onet_30_0.csv"

# NPZ
old_npz_path = dst_root / "aspectt_vectors.npz"
new_npz_path = dst_root / "aspectt_vectors_full_onet_30_0.npz"
old_csv_path.rename(new_csv_path)
old_npz_path.rename(new_npz_path)


PosixPath('/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors_full_onet_30_0.npz')

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths ---
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/")
csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks_full_onet_30_0.csv"
npz_path = dst_root / "aspectt_vectors_full_onet_30_0.npz"

csv_out = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
npz_out = dst_root / "aspectt_vectors.npz"

# --- your 40 titles (already validated present) ---
core_set = set(t.strip() for t in core_titles)

# --- load csv ---
df = pd.read_csv(csv_path, dtype=str)
title_col = "Title"

# mask: keep rows NOT in core list
keep_titles = ~df[title_col].fillna("").astype(str).str.strip().isin(core_set)

df_f = df.loc[keep_titles].copy()
df_f.to_csv(csv_out, index=False)

# --- load npz ---
data = np.load(npz_path, allow_pickle=True)

# build keep mask from npz titles
npz_titles = pd.Series(data["titles"]).astype(str).str.strip()
keep_npz = ~npz_titles.isin(core_set)
keep_npz = keep_npz.to_numpy()

# filter every aligned array (same first dimension as titles)
out = {}
n = len(npz_titles)
for k in data.files:
    arr = data[k]
    if hasattr(arr, "shape") and len(arr.shape) > 0 and arr.shape[0] == n:
        out[k] = arr[keep_npz]
    else:
        out[k] = arr  # metadata etc stays untouched

np.savez_compressed(npz_out, **out)

# --- sanity checks ---
print("Dropped from CSV:", int((~keep_titles).sum()), "kept:", len(df_f))
print("Dropped from NPZ:", int((~keep_npz).sum()), "kept:", int(keep_npz.sum()))
print("CSV/NPZ kept titles match?",
      set(df_f[title_col].astype(str).str.strip()) == set(out["titles"].astype(str)))


Dropped from CSV: 0 kept: 861
Dropped from NPZ: 0 kept: 861
CSV/NPZ kept titles match? True


In [15]:
csv_out = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
npz_out = dst_root / "aspectt_vectors.npz"

standard_df_onet_occupations_description_activities_and_tasks = pd.read_csv(csv_out)
data = npz_out
path = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/prod/embeddings/aspectt_vectors.npz")
data = np.load(path,allow_pickle=True)
print(data.files)
print('same onet titles for onet 30.0?',len(set(data['titles']).intersection(set(standard_df_onet_occupations_description_activities_and_tasks['Title']))) == 861)
print('occupations set size:',len(set(data['titles'])))

['titles', 'columns', 'levels', 'importance']
same onet titles for onet 30.0? True
occupations set size: 861
